# M7 - Generation de contenu neuro-symbolique

**Emile Jouannet** - EPITA SCIA 2026 - Intelligence Symbolique

Un LLM ecrit de bons intitules de cours mais ne garantit rien : il oublie des objectifs et se
trompe sur l'ordre des prerequis. Un solveur CSP garantit tout mais ecrit "Session 1, Session 2".

Ce notebook montre le pattern **LLM generateur + CSP verificateur** : le LLM propose, CP-SAT
verifie, et les violations repartent dans le prompt.

## 1. Le syllabus

In [1]:
import sys
sys.path.insert(0, "src")

from m7_neurosymbolic.schema import Syllabus

syllabus = Syllabus.from_json("data/syllabus.json")
for o in syllabus.objectives:
    pre = ", ".join(o.prerequisites) if o.prerequisites else "-"
    print(f"{o.id:6} {o.label:45} prerequis: {pre}")
print(f"\n{syllabus.n_sessions} sessions, duree entre {syllabus.min_duration} et {syllabus.max_duration}")

LOGIC  Logique propositionnelle et premier ordre     prerequis: -
SAT    Solveurs SAT, DPLL et CDCL                    prerequis: LOGIC
SMT    Solveurs SMT et theories (Z3)                 prerequis: SAT
CSP    Programmation par contraintes et propagation  prerequis: LOGIC
VERIF  Verification formelle par SMT                 prerequis: SMT
ARGU   Argumentation abstraite de Dung               prerequis: LOGIC
KR     Representation des connaissances et ontologies prerequis: LOGIC
NEURO  IA neuro-symbolique : LLM et solveurs         prerequis: SMT, CSP, ARGU

6 sessions, duree entre 1 et 3


La chaine `LOGIC -> SAT -> SMT -> VERIF` est transitive, et `NEURO` depend de trois objectifs.
C'est le type de contrainte sur lequel un LLM se trompe sans le signaler.

## 2. Le syllabus est-il faisable ?

A verifier avant tout appel au LLM : si les prerequis forment un cycle, aucun plan n'existe et
la boucle echouerait jusqu'au budget max sans qu'on sache pourquoi.

Chaque contrainte de prerequis est posee sous un literal d'hypothese. Si le modele est
infaisable, CP-SAT rend le sous-ensemble d'hypotheses responsable, ce qui designe les arcs
fautifs au lieu d'un simple "infaisable".

In [2]:
from m7_neurosymbolic.validator import Validator

validator = Validator(syllabus)
feasible, reasons = validator.is_instance_feasible()
print("Faisable :", feasible)

Faisable : True


In [3]:
from m7_neurosymbolic.schema import LearningObjective

# Syllabus casse volontairement : A exige B, B exige A.
cyclic = Syllabus(
    objectives=(
        LearningObjective("A", "A", ("B",)),
        LearningObjective("B", "B", ("A",)),
    ),
    n_sessions=2,
    min_duration=1,
    max_duration=2,
)
feasible, reasons = Validator(cyclic).is_instance_feasible()
print("Faisable :", feasible)
for r in reasons:
    print(" -", r)

Faisable : False
 - Prerequis insatisfiable : 'B' doit preceder 'A'.
 - Prerequis insatisfiable : 'A' doit preceder 'B'.


Le noyau d'infaisabilite pointe les deux arcs du cycle. C'est ce que CP-SAT apporte ici :
verifier est facile, **expliquer minimalement pourquoi c'est faux** ne l'est pas.

## 3. Un plan invalide, tel qu'un LLM en produit

On rejoue un plan realiste : titres corrects, mais `SMT` est enseigne avant son prerequis `SAT`,
et `KR` est oublie.

In [4]:
import json
from m7_neurosymbolic.schema import PlanCandidate

bad_plan = PlanCandidate.from_json(json.dumps({"sessions": [
    {"title": "Fondements logiques", "objectives": ["LOGIC"], "start_slot": 0, "duration": 1},
    {"title": "Theories et Z3", "objectives": ["SMT"], "start_slot": 1, "duration": 1},
    {"title": "Solveurs SAT", "objectives": ["SAT"], "start_slot": 2, "duration": 1},
    {"title": "Contraintes et argumentation", "objectives": ["CSP", "ARGU"], "start_slot": 3, "duration": 2},
    {"title": "Verification formelle", "objectives": ["VERIF"], "start_slot": 5, "duration": 1},
    {"title": "LLM et solveurs", "objectives": ["NEURO"], "start_slot": 6, "duration": 1},
]}))

result = validator.validate(bad_plan)
print(result.summary())
for v in result.violations:
    print(f"\n[{v.kind.value}] {v.explanation}")

2 violation(s): coverage, prerequisite

[coverage] Objectifs jamais couverts : ['KR']. Chacun doit apparaitre dans au moins une session.

[prerequisite] 'SMT' est enseigne au creneau 1, mais son prerequis 'SAT' l'est au creneau 2. Le prerequis doit venir avant.


Les deux erreurs sont trouvees. Un relecteur humain aurait pu rater l'inversion `SMT`/`SAT` :
le plan se lit bien.

## 4. Le feedback envoye au LLM

In [5]:
from m7_neurosymbolic.feedback import build_feedback

print(build_feedback(result))

Le plan precedent viole des contraintes dures. Corrige uniquement ces points :

[coverage]
- Objectifs jamais couverts : ['KR']. Chacun doit apparaitre dans au moins une session.
  Consigne : Ajoute les objectifs manquants a des sessions existantes.

[prerequisite]
- 'SMT' est enseigne au creneau 1, mais son prerequis 'SAT' l'est au creneau 2. Le prerequis doit venir avant.
  Consigne : Reordonne les sessions concernees sans changer leur contenu.

Garde les titres et la progression deja corrects. Renvoie le plan complet en JSON.


Trois choix, qui sont le coeur du projet :

- **cible** : seules les contraintes violees sont renvoyees, pas le syllabus entier ;
- **local** : chaque violation nomme les objectifs concernes ;
- **actionnable** : on dit quoi faire, pas seulement ce qui est faux.

Le temoin `build_naive_feedback` dit seulement "invalide", et sert a mesurer si tout ceci sert
reellement a quelque chose (section 7).

## 5. La boucle

`ScriptedGenerator` rejoue des reponses fixees : la demonstration est deterministe et ne coute
aucun appel API. Le vrai generateur (section 8) expose la meme interface.

In [6]:
from m7_neurosymbolic.generator import ScriptedGenerator
from m7_neurosymbolic.loop import run_loop

good_plan = json.dumps({"sessions": [
    {"title": "Fondements logiques", "objectives": ["LOGIC"], "start_slot": 0, "duration": 1},
    {"title": "Solveurs SAT", "objectives": ["SAT"], "start_slot": 1, "duration": 1},
    {"title": "Theories et Z3", "objectives": ["SMT"], "start_slot": 2, "duration": 1},
    {"title": "Contraintes et argumentation", "objectives": ["CSP", "ARGU"], "start_slot": 3, "duration": 2},
    {"title": "Ontologies et verification", "objectives": ["KR", "VERIF"], "start_slot": 5, "duration": 2},
    {"title": "LLM et solveurs", "objectives": ["NEURO"], "start_slot": 7, "duration": 1},
]})

generator = ScriptedGenerator([bad_plan.to_json(), good_plan])
outcome = await run_loop(syllabus, generator, max_cycles=5)

print("Converge :", outcome.converged)
print("Cycles   :", outcome.n_cycles)
print("Violations par cycle :", outcome.violation_trajectory)

Converge : True
Cycles   : 2
Violations par cycle : [2, 0]


In [7]:
for s in outcome.final_plan.sessions:
    print(f"creneau {s.start_slot}-{s.end_slot}  {s.title:35} {list(s.objectives)}")

creneau 0-1  Fondements logiques                 ['LOGIC']
creneau 1-2  Solveurs SAT                        ['SAT']
creneau 2-3  Theories et Z3                      ['SMT']
creneau 3-5  Contraintes et argumentation        ['CSP', 'ARGU']
creneau 5-7  Ontologies et verification          ['KR', 'VERIF']
creneau 7-8  LLM et solveurs                     ['NEURO']


## 6. Baseline : CP-SAT seul

In [8]:
from m7_neurosymbolic.baselines import csp_only

csp = csp_only(syllabus)
for s in csp.final_plan.sessions:
    print(f"creneau {s.start_slot}-{s.end_slot}  {s.title:12} {list(s.objectives)}")

creneau 0-1  Session 1    ['LOGIC']
creneau 1-2  Session 2    ['KR', 'SAT']
creneau 2-3  Session 3    ['ARGU', 'CSP', 'SMT']
creneau 3-4  Session 4    ['VERIF']
creneau 4-5  Session 5    ['NEURO']


Valide par construction, et pedagogiquement inutilisable : les titres ne veulent rien dire.
C'est le contraste qui justifie de garder un LLM dans la boucle.

## 7. Ce qu'il reste a mesurer

Tout ce qui precede montre le **mecanisme**. Les resultats demandent de vraies executions LLM :

- taux de convergence et nombre de cycles, sur plusieurs runs (le LLM est stochastique, une
  execution ne prouve rien) ;
- ablation feedback cible / feedback naif ;
- qualite semantique des plans finaux, qui ne se mesure pas automatiquement.

`metrics.summarize` agrege une liste de runs.

## 8. Execution reelle

Necessite `OPENAI_API_KEY` dans `.env`. Le reste du notebook tourne sans cle.

In [9]:
# from dotenv import load_dotenv
# from m7_neurosymbolic.generator import SemanticKernelGenerator
# from m7_neurosymbolic.metrics import summarize
#
# load_dotenv()
# outcomes = [await run_loop(syllabus, SemanticKernelGenerator(), max_cycles=5) for _ in range(5)]
# print(summarize(outcomes).render())